# Notebook 04 — Baseline Experiments

Runs all 5 baselines (B0–B4) on DTAH-Bench-50 and saves results.

- **B0–B3:** CPU baselines (spec generation only, no 3D mesh)
- **B4:** Full IFC-GraphRAG-DT pipeline (GPU recommended for 3D generation)

Results are saved to Google Drive for persistence across sessions.

In [ ]:
# ── Cell 1: Setup (GPU runtime recommended) ───────────────────────────────────
import os, sys, json, time
REPO = 'ifc-graphrag-dt'

if os.path.exists(REPO):
    !cd {REPO} && git pull --quiet
else:
    !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git

!bash {REPO}/colab_setup.sh

os.chdir(REPO)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

# Mount Google Drive for persistence
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/ifc_graphrag_dt_outputs'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f'Drive mounted. Outputs → {DRIVE_OUT}')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    DRIVE_OUT = 'outputs'
    print('Drive not available — saving locally.')
    os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

# Install GPU packages only in this notebook
print('Installing diffusers and torch (GPU build)...')
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 --quiet
!pip install diffusers accelerate transformers --quiet
# TripoSR: use the last known working commit with setup.py
!pip install git+https://github.com/VAST-AI-Research/TripoSR.git@main --quiet || \
 pip install huggingface_hub trimesh open3d --quiet  # fallback if TripoSR unavailable

import torch
print(f'\nGPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from pathlib import Path
import yaml
import numpy as np
from benchmark.dtah_bench import DTAHBench
from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from pipeline.layer2_spec_gen.scene_spec_generator import SceneSpecGenerator
from pipeline.baselines.b0_no_grounding import B0NoGrounding
from pipeline.baselines.b1_llm_only import B1LLMOnly
from pipeline.baselines.b2_flat_rag import B2FlatRAG
from pipeline.baselines.b3_ifc_lookup import B3IFCLookup
from evaluation.metrics.kcs_dt import KCSDTScorer

os.makedirs('outputs/scores',  exist_ok=True)
os.makedirs('outputs/specs',   exist_ok=True)
os.makedirs('outputs/meshes',  exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
print('\nAll imports OK')

In [ ]:
# ── Cell 2: Load graph and embedder ──────────────────────────────────────────
IFC_PATH    = 'benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = 'outputs/graphs/ifc_graph.json'
EMB_CACHE   = 'outputs/embedders/graph_embedder'
os.makedirs('outputs/graphs',    exist_ok=True)
os.makedirs('outputs/embedders', exist_ok=True)

if not Path(IFC_PATH).exists():
    import urllib.request
    urllib.request.urlretrieve(
        'https://github.com/buildingSMART/Sample-Test-Files/raw/master/IFC%202x3/Duplex%20Apartment/Duplex_A_20110907.ifc',
        IFC_PATH)

if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

if Path(f'{EMB_CACHE}/faiss.index').exists():
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    print('Building embedder...')
    embedder = GraphEmbedder()
    embedder.fit(G)
    embedder.save(EMB_CACHE)

with open('pipeline/configs/ifc_config.yaml') as f:
    ifc_config = yaml.safe_load(f)

print(f'Graph: {G.number_of_nodes()} nodes | Embedder: {len(embedder._node_ids)} nodes')

In [ ]:
# ── Cell 3: Initialise baselines ──────────────────────────────────────────────
TIER_DEPTHS = {1: 1, 2: 2, 3: 4}
llm_config  = {'llm_provider': 'anthropic', 'model': 'claude-sonnet-4-20250514',
               'max_tokens': 2048, 'temperature': 0.0, 'output_dir': 'outputs/specs'}
gen_config  = {'output_dir': 'outputs/meshes'}

b0 = B0NoGrounding(gen_config)
b1 = B1LLMOnly(llm_config)
b2 = B2FlatRAG(llm_config, graph_embedder=embedder)
b3 = B3IFCLookup(llm_config)
b4_spec_gen = SceneSpecGenerator(llm_config)

bench       = DTAHBench(pilot_mode=True)
all_prompts = bench.load_all()
print(f'Running on {len(all_prompts)} pilot prompts across {len(set(p["tier"] for p in all_prompts))} tiers')

In [ ]:
# ── Cell 4: Run B0, B1, B2, B3 (CPU baselines) ───────────────────────────────
baseline_records = {'b0': [], 'b1': [], 'b2': [], 'b3': []}

for i, p in enumerate(all_prompts):
    pid, prompt, tier, domain = p['id'], p['prompt'], p['tier'], p.get('domain','MEP')
    depth    = TIER_DEPTHS[tier]
    seeds    = embedder.retrieve_seeds(prompt, top_k=5)
    seed_ids = [s['node_id'] for s in seeds]
    trav     = KHopTraversal(G, max_depth=depth, bidirectional=True)
    graph_sg = trav.traverse(seed_ids).subgraph

    # B0: no grounding
    s0 = b0.run(prompt, pid)
    baseline_records['b0'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
        'entities': 0, 'relations': 0, 'baseline': 'B0', 'status': 'ok'})

    # B1: LLM only
    try:
        s1 = b1.run(prompt, pid)
        baseline_records['b1'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
            'entities': len(s1.get('entities',[])), 'relations': len(s1.get('relations',[])),
            'baseline': 'B1', 'status': 'ok'})
    except Exception as e:
        baseline_records['b1'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
            'entities': 0, 'relations': 0, 'baseline': 'B1', 'status': f'error: {e}'})

    # B2: flat RAG
    s2 = b2.run(prompt, G, pid)
    baseline_records['b2'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
        'entities': len(s2.get('entities',[])), 'relations': len(s2.get('relations',[])),
        'baseline': 'B2', 'status': 'ok'})

    # B3: IFC direct lookup
    s3 = b3.run(prompt, G, ifc_config, pid)
    baseline_records['b3'].append({'prompt_id': pid, 'tier': tier, 'domain': domain,
        'entities': len(s3.get('entities',[])), 'relations': len(s3.get('relations',[])),
        'baseline': 'B3', 'status': 'ok'})

    print(f'[{i+1:02d}/{len(all_prompts)}] {pid} B0:0e B2:{len(s2.get("entities",[]))}e B3:{len(s3.get("entities",[]))}e', end='\r')

print(f'\n✓ CPU baselines complete.')
for b, recs in baseline_records.items():
    ok = sum(1 for r in recs if r['status'] == 'ok')
    print(f'  {b.upper()}: {ok}/{len(recs)} OK')

In [ ]:
# ── Cell 5: Run B4 — IFC-GraphRAG-DT ─────────────────────────────────────────
HAS_GPU = torch.cuda.is_available()
print(f'GPU: {HAS_GPU}')

from pipeline.run_pipeline import IFCGraphRAGDT
pipeline = IFCGraphRAGDT(
    config_path='pipeline/configs/pipeline_config.yaml',
    dry_run=not HAS_GPU   # skip 3D generation on CPU
)
print(f'Pipeline mode: {"FULL (GPU)" if HAS_GPU else "DRY RUN (no 3D gen)"}')

b4_records = []
for i, p in enumerate(all_prompts):
    pid, prompt, tier = p['id'], p['prompt'], p['tier']
    t0 = time.time()
    try:
        result = pipeline.run(prompt=prompt, prompt_id=pid, tier=tier)
        b4_records.append({'prompt_id': pid, 'tier': tier, 'domain': p.get('domain','MEP'),
            'entities': len(result.spec.get('entities',[])),
            'relations': len(result.spec.get('relations',[])),
            'mesh_path': str(result.mesh_path) if result.mesh_path else None,
            'elapsed_s': round(time.time()-t0, 2), 'baseline': 'B4', 'status': 'ok'})
    except Exception as e:
        b4_records.append({'prompt_id': pid, 'tier': tier, 'domain': p.get('domain','MEP'),
            'entities': 0, 'relations': 0, 'mesh_path': None,
            'elapsed_s': round(time.time()-t0, 2), 'baseline': 'B4', 'status': f'error: {e}'})
    print(f'[{i+1:02d}/{len(all_prompts)}] {pid} — {round(time.time()-t0,1)}s', end='\r')

ok = sum(1 for r in b4_records if r['status'] == 'ok')
print(f'\n✓ B4 complete: {ok}/{len(b4_records)} OK')

In [ ]:
# ── Cell 6: Save all results ──────────────────────────────────────────────────
all_records = {**baseline_records, 'b4': b4_records}

for b_name, recs in all_records.items():
    out = f'outputs/scores/{b_name}_scores.json'
    with open(out, 'w') as f:
        json.dump(recs, f, indent=2)
    print(f'Saved: {out} ({len(recs)} records)')

# Copy to Drive
if DRIVE_OUT != 'outputs':
    import shutil
    shutil.copytree('outputs/scores', f'{DRIVE_OUT}/scores', dirs_exist_ok=True)
    if HAS_GPU:
        shutil.copytree('outputs/meshes', f'{DRIVE_OUT}/meshes', dirs_exist_ok=True)
    print(f'Results copied to Google Drive.')

In [ ]:
# ── Cell 7: Quick summary table ───────────────────────────────────────────────
print('ENTITY COUNT SUMMARY (mean per prompt, proxy for retrieval quality)')
print(f'{"Baseline":<8} {"Tier 1":>10} {"Tier 2":>10} {"Tier 3":>10}')
print('-' * 42)
for b_name, recs in all_records.items():
    means = []
    for tier in [1, 2, 3]:
        t_recs = [r for r in recs if r.get('tier')==tier and r.get('status')=='ok']
        means.append(np.mean([r.get('entities',0) for r in t_recs]) if t_recs else 0.0)
    print(f'{b_name.upper():<8} {means[0]:>10.1f} {means[1]:>10.1f} {means[2]:>10.1f}')
print('\nProceed to Notebook 05 for evaluation and paper figures.')